In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json
os.chdir("..")

In [3]:
from datasets import load_dataset

In [4]:
from cluster_intrep_repo.utils import initialize_tokenizer, tokenize_blocksworld_generation

model_id = "Qwen/QwQ-32B"
tokenizer = initialize_tokenizer(model_id)

In [103]:
dataset = load_dataset("dmitriihook/blocksworld-mystery-qwq-reasoning-parts-exploration")["train"]

(…)ery-qwq-reasoning-parts-exploration.json:   0%|          | 0.00/26.3k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [18]:
dataset_gen = load_dataset("dmitriihook/qwq-32b-planning-mystery-24k")["train"]

In [123]:
idx = 9

print(
    dataset_gen[idx]["generation"]
)

Okay, let's see. I need to solve this problem where the initial conditions are given, and I have to come up with a plan using the actions provided to reach the goal. Let me start by understanding the problem step by step.

First, let me restate the initial conditions and the goal to make sure I have them right.

**Initial Conditions:**
- Block A craves Block C (so "Object Craves other object" for A and C)
- Block D craves Block A (so D craves A)
- Harmony exists (Harmony is true)
- Planet Block B and Planet Block C (so both B and C are on their planets)
- Province Block B and Province Block D (so B and D are in their provinces)

**Goal:**
- Block A craves Block D
- Block B craves Block A
- Block D craves Block C

So I need to manipulate the actions to get these cravings and other conditions right. Let me recall the actions and their preconditions and effects.

The actions are Attack, Feast, Succumb, and Overcome. Let me list their requirements and effects again to be precise.

**Attack

In [124]:
"First, let's tackle D→B." == "First, let's tackle D→B."

True

In [125]:
import re

label = dataset[idx]["label"]

if label is not None:
    for tag, content in label.items():

        gen_ind = dataset_gen[idx]["generation"].find(content[:30])
        print(
            content[:30]
        )
        tokens = tokenize_blocksworld_generation(tokenizer, dataset_gen[idx], dataset_gen[idx]["generation"][:gen_ind])

        print(tag, len(tokens[0]), len(tokenize_blocksworld_generation(tokenizer, dataset_gen[idx])[0]))

Let me list the steps in order
final-plan-formulation-verification 12274 13196
Let me try this plan:

1. Feas
plan-generation-verification 9085 13196


In [126]:
print(dataset[idx]["label"]["plan-generation-verification"])

Let me try this plan:

1. Feast D from A (as before)
2. Succumb D
3. Attack B
4. Overcome B from A (now B craves A)
5. Attack D (to get Pain D)
6. Succumb D (to restore Province D and Harmony)
7. Now, with Harmony true, can we attack C?

Wait, to attack C, need Province C (no), so no.

Hmm.

Alternatively, after step 4 (Overcome B from A), we have:

- Province A is false (from Overcome's effect)
- Harmony is true
- B craves A


In [11]:
import re

labels_low = [
    "initial-state-understanding",
    "goal-state-understanding",
    "comparative-analysis",
    "recursive-search",
    "plan-formulation",
    "constraint-analysis",
    "stating-actions",
    "state-tracking"
]

In [12]:
def parse_sections(text:str, labels: list[str]) -> list[tuple[str, str]]:
    pattern = r'\["([^"]+)"\](.*?)\["end-section"\]'
    matches = re.findall(pattern, text, re.DOTALL)
    
    sections = []
    for label, content in matches:
        if label in labels:
            sections.append((label, content.strip()))
    
    return sections

parse_sections(dataset[idx]["label"], labels_low)

[('initial-state-understanding',
  "Okay, let's see. I need to figure out the plan to rearrange the blocks from the initial state to the goal state. Let me start by understanding the problem step by step.\n\nFirst, the initial conditions are:\n\n- Block A is clear (so nothing is on top of it)\n- Block E is clear\n- Block F is clear\n- Hand is empty\n- The stack structure is: A on B, B on C, C on D, and D is on the table. Then, E and F are both on the table."),
 ('goal-state-understanding',
  'The goal is to have:\n\n- B on top of A\n- C on top of B\n- D on top of C\n- E on top of D\n- F on top of E'),
 ('comparative-analysis',
  'Wait, so the final stack should be A at the bottom, then B, C, D, E, F on top? Or is it B on A, then C on B, etc., forming a vertical stack? Let me parse the goal correctly. The goal states each block is on top of the previous one in the sequence B -> A? Wait, no. Wait, the goal says "Block B is on top of Block A", so B is above A. Then "C is on top of B", so 

In [30]:
def map_sections_into_tokens(sections: list[tuple[str, str]], row: dict) -> list[dict]:
    # tokens = tokenize_blocksworld_generation(tokenizer, row)
    generation = row["generation"]
    section_tokens = []
    for label, content in sections[1:]:
        text_pos = generation.find(content[:300])
        if text_pos == -1:
            continue
        text_before = generation[:text_pos]
        tokens_before = tokenize_blocksworld_generation(tokenizer, row, text_before)[0, :-2]
        content_tokens = tokenizer.encode(" " + content)[:-5]
        section_tokens.append({
            "label": label,
            "pos_before": len(tokens_before),
            "pos_after": len(tokens_before) + len(content_tokens),
            "text_pos": text_pos,
            "content": content
        })

    return section_tokens

def test_sections():
    idx = 10
    generation = dataset_gen[idx]["generation"]
    sections = parse_sections(dataset[idx]["label"], labels_low)
    section_tokens = map_sections_into_tokens(sections, dataset_gen[idx])
    tokens = tokenize_blocksworld_generation(tokenizer, dataset_gen[idx])[0]

    for st in section_tokens:
        s = generation[st["text_pos"]:st["text_pos"]+300]
        dt = tokenizer.decode(tokens[st["pos_before"]:st["pos_before"]+300])
        inter_len = min(len(s[:200]), len(dt))
        if s[:inter_len] != dt[:inter_len]:
            print("****"*10)
            # print()
            print(tokenizer.tokenize(s[:inter_len]))
            print()
            print("----"*10)
            print(tokenizer.tokenize(dt[:inter_len]))
            print()

            # print their difference
            

test_sections()

****************************************
['So', 'Ġthe', 'Ġinitial', 'Ġstacks', 'Ġare', ':ĊĊ', 'Starting', 'Ġfrom', 'Ġthe', 'Ġtable', ':ĊĊ', 'E', 'Ġis', 'Ġon', 'Ġthe', 'Ġtable', '.', 'ĠOn', 'ĠE', 'Ġis', 'ĠF', '.', 'ĠOn', 'ĠF', 'Ġis', 'ĠC', '.', 'ĠOn', 'ĠC', 'Ġis', 'ĠB', '.', 'ĠOn', 'ĠB', 'Ġis', 'ĠA', '.', 'ĠOn', 'ĠA', 'Ġis', 'ĠD', '.', 'ĠAnd', 'ĠD', 'Ġis', 'Ġclear', '.', 'ĠSo', 'Ġthe', 'Ġstack', 'Ġis', 'ĠE', 'Ġ->', 'ĠF', 'Ġ->', 'ĠC', 'Ġ->', 'ĠB', 'Ġ->', 'ĠA', 'Ġ->', 'ĠD', '.', 'ĠAnd', 'ĠD', 'Ġis', 'Ġthe']

----------------------------------------
['Ġthe', 'Ġinitial', 'Ġstacks', 'Ġare', ':ĊĊ', 'Starting', 'Ġfrom', 'Ġthe', 'Ġtable', ':ĊĊ', 'E', 'Ġis', 'Ġon', 'Ġthe', 'Ġtable', '.', 'ĠOn', 'ĠE', 'Ġis', 'ĠF', '.', 'ĠOn', 'ĠF', 'Ġis', 'ĠC', '.', 'ĠOn', 'ĠC', 'Ġis', 'ĠB', '.', 'ĠOn', 'ĠB', 'Ġis', 'ĠA', '.', 'ĠOn', 'ĠA', 'Ġis', 'ĠD', '.', 'ĠAnd', 'ĠD', 'Ġis', 'Ġclear', '.', 'ĠSo', 'Ġthe', 'Ġstack', 'Ġis', 'ĠE', 'Ġ->', 'ĠF', 'Ġ->', 'ĠC', 'Ġ->', 'ĠB', 'Ġ->', 'ĠA', 'Ġ->', 'ĠD', '.', 'ĠAnd', 'ĠD', 

In [63]:
idx = 10

sections = parse_sections(dataset[idx]["label"], labels_low)
section_tokens = map_sections_into_tokens(sections, dataset_gen[idx])